# 章节练习

本练习覆盖 **模型执行与优化** 模块的核心知识点：静态 Shape 整图下沉执行流程（4.2）、动态 Shape Host 调度执行流程（4.3）、静态 Shape 执行优化技术（4.4）、动态 Shape 执行优化技术（4.5），以及贯穿全章的 **profiling 收益验证** 方法。

完成下列题目检验学习效果，如有错误，建议结合对应小节复盘，温故而知新。题目分四层：判断题、单选题、多选题、实践题。

## 一、判断题

1. （判断题）静态 Shape 模型能整图下沉的根本前提是所有 Tensor 的 shape 在编译期完全确定。

2. （判断题）整图下沉后，稳定执行阶段 Host 通常只需进行一次模型级执行触发，即可由 Device 调度模型内的多个 Task。

3. （判断题）包含 Unknown Shape 节点的完整图也能作为一个静态模型整体 Task Sink。

4. （判断题）动态 Shape 图中的每个节点都必须在每次执行时依次完成 InferShape、Tiling 和新的物理显存申请。

5. （判断题）Tiling 下沉（`ge.tiling_schedule_optimize`）仅在静态 Shape 图中生效，动态 Shape 图不适用。

6. （判断题）TensorMove 消除和 Concat No Task 需要用户手工配置选项才会生效。

7. （判断题）静态 Shape 开启单流串行执行时，在线 GE 图编译使用 `ge.enableSingleStream=true`，Ascend Graph C++ 构建向 `aclgrphBuildInitialize` 传入 `{ge::ir_option::ENABLE_SINGLE_STREAM, "true"}`，ATC 使用 `--enable_single_stream=true`。

8. （判断题）使用动态分档后，即使执行最小档位，模型内存占用仍等同于最大档位。

## 二、单选题

9. （单选题）模型下沉调度分为哪两个阶段？
    A. 图准备和图拆分
    B. 模型加载和模型执行
    C. 编译和反编译
    D. InferShape 和 Tiling

10. （单选题）整图下沉相比 Host 调度，性能收益最大的场景是？
    A. 模型是 Device Bound（算子计算很重）
    B. 模型是 Host Bound（单算子很快、算子很多，Host 下发是瓶颈）
    C. 模型只有一个算子
    D. 模型输入 shape 每次都在变化

11. （单选题）对于需要运行时 Shape 推导和 Tiling 的 Unknown Shape AI Core 节点，典型处理链路是？
    A. Tiling → InferShape → 下发 Kernel → Buffer 获取/刷新
    B. InferShape → Tiling → Buffer 获取/刷新 → 下发 Kernel
    C. Buffer 获取/刷新 → 下发 Kernel → InferShape → Tiling
    D. 下发 Kernel → InferShape → Buffer 获取/刷新 → Tiling

12. （单选题）ACL 离线模型加载（V1）中，哪个公开的 `aclmdlConfigAttr` 用于配置 workspace 的输入输出内存优化？
    A. `ACL_MDL_PRIORITY_INT32`
    B. `ACL_MDL_WORKSPACE_MEM_OPTIMIZE`
    C. `ACL_MDL_LOAD_TYPE_SIZET`
    D. `ACL_MDL_WEIGHT_PATH_PTR`

13. （单选题）以下哪个 atc 参数用于配置「仅 batch 维度变化」的动态分档？
    A. `--dynamic_image_size`
    B. `--dynamic_dims`
    C. `--dynamic_batch_size`
    D. `--input_format`

14. （单选题）离线推理时，动态 batch 模型在执行前设置真实 batch 档位应调用哪个接口？
    A. `aclmdlSetDynamicHWSize`
    B. `aclmdlSetDynamicBatchSize`
    C. `aclmdlSetInputDynamicDims`
    D. `aclmdlSetDatasetTensorDesc`

15. （单选题）动态 Shape 模型 timeline 上 Host 线很长、Device 线大量空泡，应判定为？
    A. Device Bound，优先优化算子实现
    B. Host Bound，优先减少 Host 每次重做的工作（如动态分档）
    C. 内存不足
    D. 一切正常，无需优化

## 三、多选题

16. （多选题）以下关于整图下沉与下沉收益验证的描述，哪些是正确的？
    A. 下沉把「每次执行 N 次算子级下发」转化为「加载阶段固化整图任务 + 稳定执行阶段一次模型级触发」
    B. 可用 `msprof --application` 直接拉起离线推理程序采集 timeline
    C. 下沉生效且原瓶颈是 Host 下发时，Device timeline 中由 Host 调度造成的空泡通常会减少
    D. 只要开启下沉，任何模型 E2E 耗时都必然大幅下降

17. （多选题）以下关于动态 Shape 执行的描述，哪些是正确的？
    A. 包含 Unknown Shape 节点的完整图不能作为一个静态模型整体 Task Sink
    B. 只有被编译结果判定为需要运行时 Shape/Tiling 的节点才执行相应步骤，Buffer 通常从缓存池获取或刷新
    C. GE 编译期与运行期复用同一套算子 InferShape 注册函数以保证一致
    D. 动态图中满足条件的 Known Shape 子图仍可通过静态执行器执行
    E. 动态 Shape 的时延稳定性必然优于静态 Shape

18. （多选题）以下关于静态 Shape 执行优化的描述，哪些是正确的？
    A. 静态 Shape 下 GE 可做模型级内存复用，降低显存占用
    B. 多流并发让无依赖算子在不同流上并行，跨流处插入 Event/Notify 同步
    C. RT2 StreamExecutor 开启 `gert::LoweringOption::always_zero_copy` 后，用户必须保证输出内存大小与 placement 正确，否则报错
    D. 多流一定比单流快，流越多越好

19. （多选题）以下关于动态分档与动态图兜底的描述，哪些是正确的？
    A. 动态分档把可枚举的 shape 重新变回静态 Shape 以重获优化收益
    B. 设置的档位值若未命中任何编译档位，且没有动态图兜底，执行会失败
    C. 应用侧可把未命中档位且不能安全 padding 的输入路由到独立的动态 shape 图执行
    D. 动态分档可无限增加档位且不增加编译时间和内存

20. （多选题）以下关于 profiling 收益验证的描述，哪些是正确的？
    A. GE Profiling 分层采集 API 层、Host 层、Device 层，可区分 Host 耗时与 Device 耗时
    B. 在线可通过 `ge.exec.profilingMode="1"` + `ge.exec.profilingOptions` 开启
    C. 可按 `aclgrphProfInit → aclgrphProfCreateConfig → aclgrphProfStart → 执行 → aclgrphProfStop → aclgrphProfFinalize → aclgrphProfDestroyConfig` 圈定采集区间并管理配置生命周期
    D. 判断 Host/Device Bound 与选择优化方向无关

## 四、实践题

> 实践题不设唯一答案，重在「按课程步骤跑通闭环 + 记录现象 + 说明定位入口」。所有对照实验应使用相同输入、warm-up 次数和稳定态采集迭代数，并把首次加载/初始化与稳定执行分开。可结合 answer 中的参考思路自评。

21. （跑通闭环）选取一个 ResNet 类模型，用 atc 编译为**静态 Shape** OM（`--input_shape` 全确定值），让推理程序先 warm-up、再重复执行并用 `msprof --application=...` 采集稳定态 timeline。请：
    - 写出你的 atc 编译命令与 msprof 采集命令；
    - 结合 `--runtime-api=on` 指出每次执行 Host 侧的模型触发 / Runtime Launch 数，并与 Device 上的多个算子 Task 区分；
    - 观察 Device 是否连续，并说明空泡是否由 Host 调度造成；
    - 据此说明该模型是否已走整图下沉。

22. （改动实验）把同一模型改为**完全动态 Shape**（`--input_shape` 某维为 `-1`，不配档位）重新编译；ACL 执行前用 `aclmdlSetDatasetTensorDesc` 设置真实输入描述，执行后用 `aclmdlGetDatasetTensorDesc` 获取动态输出描述。使用 `--ge-api=l1 --runtime-api=on` 采集并与第 21 题对比：
    - 哪些节点出现了运行时 InferShape / Tiling，Buffer 获取或刷新耗时有何变化？
    - Device 空泡是否增多？
    - 据此判定动态版本是 Host Bound 还是 Device Bound，并说明判定依据。

23. （改动实验）在第 22 题动态模型的基础上，改用**动态分档**（如 `--dynamic_batch_size="1,8,16"`）编译；通过 `ACL_DYNAMIC_TENSOR_NAME` 获取动态档位输入下标，执行前用 `aclmdlSetDynamicBatchSize` 设置命中档位。按相同 warm-up、输入和迭代口径采集 timeline：
    - 对比「完全动态」与「分档命中」的 GE Host 阶段、Host Runtime 模型触发、Device Task、空泡和 E2E 耗时；
    - 说明分档带来的预期收益来源，并用实际数据判断收益是否成立（提示：命中档位走静态子图、可下沉）。

24. （改动实验）选择一种明确的执行路径验证**零拷贝 / I/O 内存优化**，不要混用不同路径的配置语义：
    - ACL V1 路径：通过 `aclmdlSetConfigOpt(..., ACL_MDL_WORKSPACE_MEM_OPTIMIZE, ...)` 配置输入输出内存优化，使用满足要求的 Device I/O，并以 `--runtime-api=on` 对比模型执行窗口内的 memcpy；
    - 区分业务自身不可避免的 H2D/D2H 与模型侧额外 D2D 回退拷贝，说明为什么地址相同不能单独证明 Kernel 直接写入；
    - 扩展说明：RT2 StreamExecutor 的 `gert::LoweringOption::always_zero_copy` 与 ACL V1 路径有何不同？开启后调用方必须满足哪些输出内存约束？

25. （定位题）某动态 Shape 推理服务延迟偏高且抖动大。请给出你的排查路径：
    - 用什么采集项和指标先判断瓶颈在 Host 还是 Device？
    - 若确认是 Host Bound，你会优先尝试哪些优化手段（至少两项）？涉及 shape 归一 / padding 时需说明语义安全边界；
    - 说明为什么“为未命中档位准备独立动态图兜底”属于正确性与可用性策略，而不能单独算作 Host Bound 性能优化。

**执行以下代码获取答案。**

In [ ]:
!cat ./answer/04.06_answer.txt